# Multi-dataset: TabPFN then LoCalPFN (sequential)

This notebook runs both methods across all datasets defined in `configs/datasets.yaml`, one after the other.

- First: TabPFN across all datasets.
- Second: LoCalPFN across all datasets.

Outputs and a summary CSV are written under `notebooks/` by default.

In [1]:
# Keep your existing env caps
import os, sys, site, torch
os.environ['PYTHONNOUSERSITE'] = '1'
usr = site.getusersitepackages(); sys.path = [p for p in sys.path if p != usr]

# Cap threads broadly
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

# If you still see OpenMP warnings/hangs, you can try (may slow MKL but stabilizes):
# os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'

# Optional: allow duplicate OpenMP (stops hard-crash; you already set this)
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

# Also cap PyTorch intra-op threads
torch.set_num_threads(1)
print("Torch threads set to 1")

Torch threads set to 1


In [2]:
import sys
from pathlib import Path

def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / 'med3pipe').is_dir():
            if str(base) not in sys.path:
                sys.path.insert(0, str(base))
            print('Added repo root to sys.path:', base)
            return base
    raise RuntimeError("Could not locate 'med3pipe/' in current or parent directories.")

repo_root = _add_repo_root_to_sys_path()


Added repo root to sys.path: C:\Users\cahel\Desktop\Med3Tab-PFN


In [3]:
from pathlib import Path
from med3pipe.pipelines import run_multi_dataset
from med3pipe.tabular.localpfn import LocalPFNConfig

# Use repo_root detected above
config_path = repo_root / 'configs' / 'datasets.yaml'
outputs_base = repo_root / 'notebooks'

print('Config path:', config_path)
print('Exists:', config_path.exists())
print('Outputs base:', outputs_base)

c:\Users\cahel\.conda\envs\sammed3d\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config path: C:\Users\cahel\Desktop\Med3Tab-PFN\configs\datasets.yaml
Exists: True
Outputs base: C:\Users\cahel\Desktop\Med3Tab-PFN\notebooks


## Run TabPFN across all datasets

In [6]:
from med3pipe.pipelines import run_multi_tabpfn

res_tab = run_multi_tabpfn(
    config_path=config_path,
    outputs_base_dir=outputs_base,
    # tabpfn_clf_kwargs={"N_ensemble_configurations": 16},  # optional, if supported
)
print(res_tab["summary_df"])

Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...


KeyboardInterrupt: 

## Run LoCalPFN across all datasets

In [ ]:
from med3pipe.pipelines import run_multi_localpfn

res_loc = run_multi_localpfn(
    config_path=config_path,
    outputs_base_dir=outputs_base,
    local_k=8,
    local_fit_adapter=True,
    local_adapter_epochs=8,
    local_adapter_num_queries=150,
)
print(res_loc["summary_df"])

Prepared 25 cases ...
Prepared 50 cases ...
Prepared 75 cases ...
Prepared 100 cases ...
Prepared 125 cases ...
Prepared 150 cases ...
Prepared 175 cases ...
Prepared 200 cases ...
Prepared 225 cases ...


## Combined summary

In [ ]:
import pandas as pd
summary = pd.concat([res_tab['summary_df'], res_loc['summary_df']], ignore_index=True)
summary
